In [1]:
%matplotlib widget
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

import riemann_utils.covariances as rutils 
import riemann_utils.plots as rplt
import riemann_utils.matrix_functions as mrtrf

import py_utils.data_managment as dtmn
import py_utils.signal_processing as sgnpr

from pyriemann.utils.distance import distance_riemann
from scipy.io import loadmat

from IPython.display import clear_output

from scripts.processing_centroids import extract_run_centroids, center_covariances, extract_day_centroids
from scripts.processing_oldDataset import processDataset_20192020, processDataset_calibrations, extract_model_centroids, processDataset_PSD_20192020, processDataset_PSD_calibrations
from scripts.processing_newDataset import processDataset_20232024, concatenate_cov_datasets

import scripts.processing_oldDataset as oldp
from importlib import reload
import os
# reload(dtmn)
# reload(rplt)
# reload(sgnpr)
# reload(mrtrf)

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import pandas as pd

In [ ]:
filter_order = 4

windowsLength = 1
windowsShift = 1/16

bandranges = [[8, 26]]
classes = [771, 773]

fs = 512

doNormalize = True

doLogBandPower = False
applyLaplacian = True

pathData = '/home/palatella/workspace/cybathlon_user/'
pathImages = '/home/palatella/workspace/cybathlon_user/images/'

days_modelCreation = ['02/05/2019','09/07/2019','27/10/2020']


if doLogBandPower: print('USING LOG BAND POWER')
if doNormalize: print('USING NORMALIZATION')
if applyLaplacian: print('USING LAPLACIAN')

USING NORMALIZATION
USING LAPLACIAN


In [ ]:
file_path = f'{pathData}events_labels_dataset_user_20192020.mat'
proc_data = loadmat(file_path)

procLabels = dtmn.fix_mat(proc_data['labels'])
procVector = dtmn.fix_mat(proc_data['procVector'])
runs_labels = dtmn.fix_mat(proc_data['runs_labels'])
complete_runs_labels = dtmn.fix_mat(proc_data['complete_runs_labels'])
validRuns = np.array([x[0] for x in proc_data['validRuns']])
completeValidRuns = np.array([x[0] for x in proc_data['completeValidRuns']])

for idx,rLbl in enumerate(runs_labels):
    if len(rLbl[0]) > 1:
        t_lbl = np.squeeze(rLbl[0]).astype(float) 
        t_lbl[t_lbl==1] = 773
        t_lbl[t_lbl==2] = 771
        runs_labels[idx][0] = t_lbl

for idx,rLbl in enumerate(complete_runs_labels):
    if len(rLbl[0]) > 1:
        t_lbl = np.squeeze(rLbl[0]).astype(float) 
        t_lbl[t_lbl==1] = 773
        t_lbl[t_lbl==2] = 771
        complete_runs_labels[idx][0] = t_lbl


channels = np.array(['FC3', 'FC1', 'FCZ', 'FC2', 'FC4', 'C3', 'C1', 'CZ', 'C2', 'C4','CP1', 'CPZ', 'CP2'])

# PSD


In [ ]:
import mat73
psdSettings = {}
psdSettings['psdWlength'] = 0.5
psdSettings['wshift'] = 0.0625
psdSettings['mlength'] =  1
psdSettings['pshift'] = 0.25
psdSettings['SampleRate'] = 512
file_path = f'{pathData}dataset_user_20192020.mat'
dataset = mat73.loadmat(file_path)

print('- Dataset loaded')

dataset = dataset['data']
eeg = dataset['eeg']
ev = dataset['events']       
labels = dataset['labels']

# # processing events ____________________________________________________________________________________________________________
events = pd.DataFrame(ev)
unique_events = np.unique(events.TYP)
off_events = unique_events[unique_events>0x8000]

if 781+0x8000 in off_events:
    off = 781+0x8000
    on = off-0x8000
    idx_off_ev = events[events.TYP==off].index.to_numpy()
    idx_on_ev = events[(events.TYP==on) & (events.DUR==0)].index.to_numpy()
    pos_on = events.POS[idx_on_ev].values
    for i in idx_off_ev:
        pos_off = events.POS[i]
        idx = np.where(pos_on<pos_off)[0][-1]
        ev_idx = idx_on_ev[idx]
        events.DUR[ev_idx] = pos_off-pos_on[idx]

if 1024+0x8000 in off_events:
    eog_second = 2
    dur = eog_second*fs
    events.DUR[events.TYP==1024] = dur

events.drop(events[events.TYP>0x8000].index, inplace=True)
events.DUR[np.isnan(events.DUR)] = 0 

events_code = {'hands':         773,
            'feet':          771,
            'PadLeft':       201,
            'PadLight':      202,
            'PadRight':      203,
            'PadNone':       204,
            'CommandLeft':   101,
            'CommandLight':  102,
            'CommandRight':  103,
            'cont_fdback':   781,
            'fixation':      786,
            'race_start':    800,
            'hit':           897,
            'miss':          898,
            'eog':           1024,
            '2020_Right':    1670,
            '2020_Left':     1672}

dict_lambda = {events_code['hands']:        events_code['hands'],
            events_code['feet']:         events_code['feet'],
            events_code['PadLeft']:      events_code['hands'],
            events_code['PadLight']:     np.nan,
            events_code['PadRight']:     events_code['feet'],
            events_code['PadNone']:      np.nan,
            events_code['CommandLeft']:  events_code['hands'],
            events_code['CommandLight']: np.nan,
            events_code['CommandRight']: events_code['feet'],
            events_code['cont_fdback']:  events_code['cont_fdback'],
            events_code['fixation']:     events_code['fixation'],
            events_code['race_start']:   np.nan,
            events_code['hit']:          events_code['hit'],
            events_code['miss']:         events_code['miss'],
            events_code['eog']:          events_code['eog'],
            events_code['2020_Right']:   events_code['feet'],
            events_code['2020_Left']:    events_code['hands']}

events['TYP'] = events['TYP'].apply(lambda x: dict_lambda[x])
events.drop(events[np.isnan(events['TYP'])].index, inplace=True)
events.reset_index(drop=True, inplace=True)
events.columns = events.columns.str.lower()


# # processing channels ____________________________________________________________________________________________________________
ch_labels = {0:np.array(['FZ', 'FC3', 'FC1', 'FCZ', 'FC2', 'FC4',  'C3',  'C1', 'CZ',  'C2',  'C4', 'CP3', 'CP1', 'CPZ', 'CP2', 'CP4' ]),
                    1:np.array(['FZ', 'FC3', 'FC1', 'FCZ', 'FC2', 'FC4',  'C3',  'C1','CZ',  'C2',  'C4', 'EOG', 'CP1', 'CPZ', 'CP2', 'EOG' ])}

channels_to_remove = ['FZ']

_,idx,_ = np.intersect1d(ch_labels[0],ch_labels[1],return_indices=True)
idx.sort()
channels = ch_labels[0][idx]

idx_ch = [list(range(0,len(ch_labels[0]))), idx]
laplacian_mask = []

path = '/home/palatella/workspace/cap_utils/laplacian16.mat'
lap = loadmat(path)
laplacian_mask.append(lap['lapmask'])

path = '/home/palatella/workspace/cap_utils/laplacian16_IntrscEOG.mat'
lap = loadmat(path)
laplacian_mask.append(lap['lapmask'])

# idx_final_eeg = idx
# _,idx,_ = np.intersect1d(ch_labels[0],channels_to_remove,return_indices=True)
# idx_final_eeg = np.delete(idx_final_eeg,idx)
# channels = np.delete(channels,idx)


# # processing eeg  _______________________________________________________________________________________________________________
print('- Applying laplacian')
for eog_idx in ch_labels.keys():
    idx_eog = labels['EOGk']==eog_idx
    eeg[np.ix_(idx_eog, idx_ch[eog_idx])] = eeg[np.ix_(idx_eog, idx_ch[eog_idx])] @ laplacian_mask[eog_idx]

- Dataset loaded
- Applying laplacian


In [15]:
reload(oldp)

<module 'scripts.processing_oldDataset' from '/home/palatella/workspace/Riemann-centroids-study/scripts/processing_oldDataset.py'>

In [ ]:
fisher, cva, freqs = oldp.get_FisherScore_perDay(eeg, labels, events, psdSettings['psdWlength'], psdSettings['wshift'], psdSettings['pshift'], mlength=psdSettings['mlength'], fs=psdSettings['SampleRate'])


In [ ]:
import mat73
psdSettings = {}
psdSettings['psdWlength'] = 0.5
psdSettings['wshift'] = 0.0625
psdSettings['mlength'] =  1
psdSettings['pshift'] = 0.25
psdSettings['SampleRate'] = 512
file_path = f'{pathData}dataset_user_20192020.mat'
dataset = mat73.loadmat(file_path)
psd, freq, psd_events, runVector, eogVector, dayVector, modalityVector, protocolVector, channelBandDict = processDataset_PSD_20192020(dataset, psdSettings['psdWlength'], psdSettings['wshift'], psdSettings['pshift'], mlength=psdSettings['mlength'], fs=psdSettings['SampleRate'])

savemat_path = f'{pathData}psd_user_20192020.mat'
# dtmn.save(savemat_path, {'psd': psd, 'freq': freq, 'psd_events': psd_events.to_numpy(), 'column_names': psd_events.columns.values, 'runVector': runVector, 'eogVector': eogVector, 'dayVector': dayVector, 'modalityVector': modalityVector, 'protocolVector': protocolVector, 'channelBandDict': channelBandDict})

mat_data = loadmat(f'{pathData}psd_user_20192020.mat')
psd = mat_data['psd']
freq = mat_data['freq'].squeeze()
runVector = dtmn.fix_mat(mat_data['runVector']).squeeze()
eogVector = dtmn.fix_mat(mat_data['eogVector']).squeeze()
dayVector = dtmn.fix_mat(mat_data['dayVector']).squeeze()
modalityVector = dtmn.fix_mat(mat_data['modalityVector']).squeeze()
protocolVector = dtmn.fix_mat(mat_data['protocolVector']).squeeze()
channelBandDict = dtmn.fix_mat(mat_data['channelBandDict'])

In [6]:
# isCFeedback = procVector['isFeedback'].astype(bool) & ((protocolVector==2) | (protocolVector==3)) # only evaluations
# validRuns = np.unique(runVector[isCFeedback]).astype(int)


# fisher, cva = sgnpr.compute_fisher_score(psd, freq, classes, runVector, runs_labels, isCFeedbackVector=isCFeedback, validRuns=validRuns)

In [7]:
# cal_psd, _, cal_psd_events, _ = processDataset_PSD_calibrations(days_modelCreation, psdSettings['psdWlength'], psdSettings['wshift'], psdSettings['pshift'], mlength=psdSettings['mlength'], fs=psdSettings['SampleRate'])

In [8]:
# def lin_ccc(x, y):
#     """Compute Lin’s Concordance Correlation Coefficient between two 1D arrays."""
#     x_mean = np.mean(x)
#     y_mean = np.mean(y)
#     s_x = np.var(x)
#     s_y = np.var(y)
#     cov_xy = np.cov(x, y, bias=True)[0, 1]
    
#     return (2 * cov_xy) / (s_x + s_y + (x_mean - y_mean)**2)

# n_Runs = len(validRuns)

# ccc_scores_fisher = np.zeros(n_Runs)
# ccc_scores_cva = np.zeros(n_Runs)

# fisher_similarities = np.zeros(n_Runs)
# cva_similarities = np.zeros(n_Runs)

# for i in range(0,n_Runs):
#     if i == 0:
#         j = i
#     else:   
#         j = i - 1
#     ccc_scores_fisher[i] = lin_ccc(fisher[:, :, i].flatten(), fisher[:, :, j].flatten())
#     ccc_scores_cva[i] = lin_ccc(cva[:, :, i].flatten(), cva[:, :, j].flatten())

#     fisher_similarities[i] = np.corrcoef(fisher[:, :, i].flatten(), fisher[:, :, j].flatten())[0, 1]
#     cva_similarities[i] = np.corrcoef(cva[:, :, i].flatten(), cva[:, :, j].flatten())[0, 1]

In [9]:
# cal_fisher = np.empty(len(days_modelCreation), dtype=object)
# cal_cva = np.empty(len(days_modelCreation), dtype=object)

# for k in range(len(days_modelCreation)):
#     data = cal_psd[k]
#     ev = cal_psd_events[k]
#     cal_runVector = np.ones(data.shape[0])
#     cal_runs_labels = np.empty(1, dtype=object)
#     cal_runs_labels[0] = np.zeros(data.shape[0])

#     for idx, cfRow in ev[ev.typ==781].iterrows():
#         cue = ev.loc[idx-1,'typ']
#         cal_runs_labels[0][cfRow.pos:cfRow.pos+cfRow.dur] = cue

#     cal_isCFeedbackVector = (cal_runs_labels[0] == 771) | (cal_runs_labels[0] == 773)
#     cal_runs_labels[0] = cal_runs_labels[0][cal_isCFeedbackVector].astype(int)

#     cal_fisher[k], cal_cva[k] = sgnpr.compute_fisher_score(data, freq, classes, cal_runVector, cal_runs_labels, isCFeedbackVector=cal_isCFeedbackVector)
    

In [10]:
# recalibrations = days_modelCreation[1:]
# rec_idx = np.array([next((x_dates[i] for i, s in enumerate(dates) if s == pattern), -1) for pattern in recalibrations])
# rec_zones = np.concatenate(([0], rec_idx, [n_Runs]))

In [11]:
# ccc_scores_fisher_cal = np.zeros(n_Runs)
# ccc_scores_cva_cal = np.zeros(n_Runs)

# fisher_similarities_cal = np.zeros(n_Runs)
# cva_similarities_cal = np.zeros(n_Runs)

# for i in range(0,n_Runs):
#     nRange = np.where(rec_zones <= i)[0][-1]

#     fisher_ref = cal_fisher[nRange]
#     cva_ref = cal_cva[nRange]
                                      
#     ccc_scores_fisher_cal[i] = lin_ccc(fisher[:, :, i].flatten(), fisher_ref.flatten())
#     ccc_scores_cva_cal[i] = lin_ccc(cva[:, :, i].flatten(), cva_ref.flatten())

#     fisher_similarities_cal[i] = np.corrcoef(fisher[:, :, i].flatten(), fisher_ref.flatten())[0, 1]
#     cva_similarities_cal[i] = np.corrcoef(cva[:, :, i].flatten(), cva_ref.flatten())[0, 1]

In [12]:
# plt.figure(figsize=(10,6))
# # plt.plot(range(n_Runs), fisher_similarities_cal, label='Fisher SIMILARITY')
# plt.plot(range(n_Runs), cva_similarities_cal, label='CVA similaruty')
# # plt.plot(range(n_Runs), ccc_scores_fisher_cal, label='Fisher CCC (cal)')
# plt.plot(range(n_Runs), ccc_scores_cva_cal, label='CVA CCC (cal)')
# plt.legend()
# plt.show()

In [13]:
# cvaSimK = np.empty(cva_similarities.shape[0]*4)
# cvaCCCK = np.empty(cva_similarities.shape[0]*4)
# calCvaSimK = np.empty(cva_similarities.shape[0]*4)
# calCvaCCCK = np.empty(cva_similarities.shape[0]*4)
# for k in range(0, cva_similarities.shape[0]):
#     cvaSimK[k*4:(k+1)*4] = cva_similarities[k]
#     cvaCCCK[k*4:(k+1)*4] = ccc_scores_cva[k]
#     calCvaSimK[k*4:(k+1)*4] = cva_similarities_cal[k]
#     calCvaCCCK[k*4:(k+1)*4] = ccc_scores_cva_cal[k]

# completeDataFrame['cvaSiml'] = cvaSimK
# completeDataFrame['cvaCCC'] = cvaCCCK
# completeDataFrame['calCvaSiml'] = calCvaSimK
# completeDataFrame['calCvaCCC'] = calCvaCCCK

# OLD DATASET 2019-2020

In [10]:
resultsDf = pd.DataFrame()

In [ ]:
# saveName = 'covs_user.mat' if not doLogBandPower else 'covs_logBandPower_user.mat'
saveName = 'covs_user_1band_v2.mat'

if not applyLaplacian: saveName = saveName.replace('user', 'user_noLap')

processDataset_20192020(pathData, doLogBandPower, bandranges, filter_order, windowsLength, windowsShift, saveData=True, applyLaplacian=applyLaplacian, saveName=saveName, fs=fs)

data = loadmat(f'{pathData}{saveName}')

covs = data['covs']
columns_name = [x[0] for x in data['column_names'][0]]
cov_events = pd.DataFrame(data['cov_events'], columns=columns_name)
utilsVector = dtmn.fix_mat(data['utilsVector'][0])
n_samples = covs.shape[1]

n_days = np.max(cov_events['day'])
n_total_runs = np.max(cov_events['run'])
days = np.array([utilsVector['daysLabel'][i::n_days] for i in range(n_days)])
days = np.array([x[-2:]+'/'+x[-4:-2]+'/'+x[0:4] for x in days])
utilsVector['isCFeedback'] = procVector['isFeedback'] 


accuracy_file = loadmat(f'{pathData}F1.accuracy_whi.mat')
accuracy_file = dtmn.fix_mat(accuracy_file['accuracy'])
performances = accuracy_file['race']
run_accuracy = performances['acc'][:-1]
run_rejection = performances['rej'][:-1]
accuracy_id_run = performances['Id'][:-1]
x_accuracy = np.array(range(0, len(accuracy_id_run)))

In [ ]:
n_samples = covs.shape[1]
n_days = np.max(cov_events['day'])

dayVector = utilsVector['day']-1
runVector = utilsVector['run']
protocolVector = utilsVector['protocol']
isCFeedback = utilsVector['isCFeedback'].astype(bool)

isCFeedback = isCFeedback & ((protocolVector==2) | (protocolVector==3)) # only evaluations
validRuns = np.unique(runVector[isCFeedback]).astype(int)

# filenameMatrices = 'centering_logBandPower_matrices.mat' if doLogBandPower else 'centering_matrices.mat'
# if not applyLaplacian: saveName = saveName.replace('matrices', 'matrices_noLap')
filenameMatrices = 'centering_matrices_1band_v2.mat'
if not applyLaplacian: saveName = saveName.replace('matrices', 'matrices_noLap')

if os.path.exists(f'{pathData}{filenameMatrices}'):
    saveTransformMatrices = False
    matrices = loadmat(f'{pathData}{filenameMatrices}')
else:
    saveTransformMatrices = True
    matrices = {}
    matrices['mean_cov'] = None
    matrices['inv_sqrt_mean_cov'] = None

covs_centered, _, _ = center_covariances(covs, dayVector, isCFeedback, referenceDay=0, saveTransformMatrices=saveTransformMatrices, pathData=pathData, saveName=filenameMatrices, mean_cov=matrices['mean_cov'], inv_sqrt_mean_cov=matrices['inv_sqrt_mean_cov'])

# saveName = 'run_centroids_user.mat' if not doLogBandPower else 'run_centroids_logBandPower_user.mat'
saveName = 'run_centroids_user_1band_v2.mat' if not doLogBandPower else 'run_centroids_logBandPower_user.mat'

if not applyLaplacian: saveName = saveName.replace('user', 'user_noLap')

extract_run_centroids(covs_centered, cov_events, validRuns, classes, runVector, isCFeedback, runs_labels, saveData=True, pathData=pathData, doLogBandPower=doLogBandPower, saveName=saveName)



In [ ]:
# import scripts.processing_centroids as proc_cent
# reload(proc_cent)
# saveName = 'day_centroids_user.mat' if not doLogBandPower else 'day_centroids_logBandPower_user.mat'
# if not applyLaplacian: saveName = saveName.replace('user', 'user_noLap')
# proc_cent.extract_day_centroids(covs_centered, cov_events, validRuns, classes, runVector, dayVector, isCFeedback, runs_labels, saveData=True, pathData=pathData, doLogBandPower=doLogBandPower, saveName=saveName)

In [ ]:
saveName = 'run_centroids_user_1band_v2.mat'
if not applyLaplacian: saveName = saveName.replace('user', 'user_noLap')

run_centroids = loadmat(f'{pathData}{saveName}')

ref_ForAngle = run_centroids['ref_angle']
d_ToEye = run_centroids['d_ToEye']
std_eyeDistance = run_centroids['std_eyeDistance']
std_centroids = run_centroids['std_centroids']
mAbsDev_centroids = run_centroids['mAbsDev_centroids']
run_centroids = run_centroids['run_centroids']

In [8]:
run_starts = np.unique(runVector, return_index=True)
idx_good_runs = np.intersect1d(run_starts[0], validRuns, return_indices=True)[1]
run_starts = run_starts[1][idx_good_runs]
dates_info = np.unique(dayVector[run_starts], return_index=True)
dates = days[dates_info[0].astype(int)]
x_dates = dates_info[1]

In [19]:
run_centroids.shape

(1, 162, 2, 13, 13)

# EVALUATE CENTROIDS MOVEMENTS

In [9]:
if doNormalize:     d_ToEye /= mAbsDev_centroids  # because it's between a distribution and a point

In [10]:
d_betweenClasses = np.full((2, run_centroids.shape[1], len(classes), len(classes)), np.nan)

for cl in range(len(classes)):
    for cl2 in range(len(classes)):
        if cl == cl2:
            d_betweenClasses[:,0,cl,cl2] = 0
            d_betweenClasses[:,1:,cl,cl2] = distance_riemann(run_centroids[:,1:,cl], run_centroids[:,:-1,cl2])
            if doNormalize:     d_betweenClasses[:,1:,cl,cl2] /= (mAbsDev_centroids[:,1:,cl]+mAbsDev_centroids[:,:-1,cl])/2
        else:
            d_betweenClasses[:,:,cl,cl2] = distance_riemann(run_centroids[:,:,cl], run_centroids[:,:,cl2])
            if doNormalize:     d_betweenClasses[:,:,cl,cl2] /= (mAbsDev_centroids[:,:,cl]+mAbsDev_centroids[:,:,cl2])/2

# EVALUATE CENTROIDS ANGLES

In [ ]:
# from scipy.io import savemat
# matrix_angle = mrtrf.compute_runMatrix_angles(run_centroids)
# savemat(f'{pathData}matrix_angle_user2019.mat', {'matrix_angle': matrix_angle})


In [11]:
center_point = np.eye(run_centroids.shape[-1])

angles_to_zero = np.full((run_centroids.shape[0], run_centroids.shape[1], len(classes)), np.nan)
classes_angles = np.full((run_centroids.shape[0], run_centroids.shape[1]), np.nan)
for run_idx in np.ndindex(angles_to_zero.shape):
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = ref_ForAngle[run_idx[0]]
    vec = run_centroids[run_idx]
    angles_to_zero[run_idx[0], run_idx[1], run_idx[2]], _ = mrtrf.angle_between_matrices(base, vec, center_point)

# angles_to_zero = mrtrf.evaluate_negative_angles(angles_to_zero, run_centroids, center_point, positiveCen=run_centroids[0,0,0], positiveAngl=angles_to_zero[0,0,0])

In [12]:
angles_to_otherClass = np.full((run_centroids.shape[0], run_centroids.shape[1]), np.nan)
for run_idx in np.ndindex(angles_to_otherClass.shape[:2]):
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = run_centroids[run_idx[0], run_idx[1], 0]
    vec = run_centroids[run_idx[0], run_idx[1], 1]
    angles_to_otherClass[run_idx[0], run_idx[1]], _ = mrtrf.angle_between_matrices(base, vec, center_point)
from scipy.io import savemat
# savemat(f'{pathData}completeResults_20192020_t_betwCl_angles.mat', {'angles_to_otherClass': angles_to_otherClass})

In [13]:
angles_to_before = np.zeros((run_centroids.shape[0], run_centroids.shape[1], len(classes)))
for run_idx in np.ndindex(angles_to_before.shape):
    if run_idx[1] == 0:
        continue
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = run_centroids[run_idx[0], run_idx[1]-1, run_idx[2]]
    vec = run_centroids[run_idx]
    angles_to_before[run_idx[0], run_idx[1], run_idx[2]], _ = mrtrf.angle_between_matrices(base, vec, center_point)
    
# from scipy.io import savemat
# savemat(f'{pathData}completeResults_20192020_t_before_angles.mat', {'angles_to_before': angles_to_before})

/home/palatella/workspace/Riemann-centroids-study/riemann_utils/matrix_functions.py:21: RuntimeWarning: invalid value encountered in arccos
  angle = np.arccos(cos_theta)


In [ ]:
# from scipy.io import savemat
# savemat(f'{pathData}completeResults_20192020_centroidsMetrics_winterCenter.mat', {
#     'd_ToEye': d_ToEye,
#     'std_eyeDistance': std_eyeDistance,
#     'std_centroids': std_centroids,
#     'mAbsDev_centroids': mAbsDev_centroids,
#     'd_betweenClasses': d_betweenClasses,
#     'angles_to_zero': angles_to_zero,
#     'angles_to_calibration': angles_to_calibration,
#     'angles_to_otherClass': angles_to_otherClass,
#     'angles_to_before': angles_to_before
# })


In [14]:
rows = []
for nrun, band, task in np.ndindex((d_ToEye.shape[1], d_ToEye.shape[0], d_ToEye.shape[2])):
    rows.append({
        'nRun': nrun,
        'band': band,
        'task': task,
        'runId' : validRuns[nrun],
        'nDay': dayVector[run_starts[nrun]].astype(int),
        'dayLabel': dates[np.where(x_dates<=nrun)[0][-1]],

        'distance' : d_ToEye[band, nrun, task],
        'angle' : angles_to_zero[band, nrun, task],

        'btw_distance': d_betweenClasses[band, nrun, 0, 1],
        'btw_angle': angles_to_otherClass[band, nrun],

        'wth_distance': d_betweenClasses[band, nrun, task, task],
        'wth_angle': angles_to_before[band, nrun, task],
     
        'accuracy': run_accuracy[nrun],
        'rejection': run_rejection[nrun]
    })

resultsDf = pd.DataFrame(rows)

In [ ]:
resultsDf.to_csv(f'/home/palatella/workspace/cybathlon_results/results_user_2019_1band_v2.csv', index=False)

# WINTER REFERENCE

In [ ]:
idxWinterRun = 75
winterRun = validRuns[idxWinterRun]

# saveTransformMatrices = True
# matrices = {}
# matrices['mean_cov'] = None
# matrices['inv_sqrt_mean_cov'] = None
# filenameMatrices = 'centering_logBandPower_matrices_winterStop.mat' if doLogBandPower else 'centering_matrices_winterStop.mat'
# if not applyLaplacian: saveName = saveName.replace('matrices', 'matrices_noLap')

# covs_centered, _, _ = center_covariances(covs, dayVector, isCFeedback, referenceDay=dayVector[runVector>=winterRun][0], saveTransformMatrices=saveTransformMatrices, 
#                                          pathData=pathData, saveName=filenameMatrices, mean_cov=matrices['mean_cov'], inv_sqrt_mean_cov=matrices['inv_sqrt_mean_cov'])

# saveName = 'run_centroids_user_winterCenter.mat' if not doLogBandPower else 'run_centroids_logBandPower_user_winterCenter.mat'
# if not applyLaplacian: saveName = saveName.replace('user', 'user_noLap')

# # extract_run_centroids(covs_centered, cov_events, validRuns, classes, runVector, isCFeedback, runs_labels, runRef=winterRun, saveData=True, pathData=pathData, doLogBandPower=doLogBandPower, saveName=saveName)

In [ ]:
saveName = 'run_centroids_user_winterCenter.mat' if not doLogBandPower else 'run_centroids_logBandPower_user_winterCenter.mat'
if not applyLaplacian: saveName = saveName.replace('user', 'user_noLap')

run_centroids = loadmat(f'{pathData}{saveName}')

ref_ForAngle = run_centroids['ref_angle']
d_ToEye = run_centroids['d_ToEye']
std_eyeDistance = run_centroids['std_eyeDistance']
std_centroids = run_centroids['std_centroids']
mAbsDev_centroids = run_centroids['mAbsDev_centroids']
run_centroids = run_centroids['run_centroids']

In [65]:
if doNormalize:     d_ToEye /= mAbsDev_centroids  # because it's between a distribution and a point


d_betweenClasses = np.full((2, run_centroids.shape[1], len(classes), len(classes)), np.nan)

for cl in range(len(classes)):
    for cl2 in range(len(classes)):
        if cl == cl2:
            d_betweenClasses[:,0,cl,cl2] = 0
            d_betweenClasses[:,1:,cl,cl2] = distance_riemann(run_centroids[:,1:,cl], run_centroids[:,:-1,cl2])
            if doNormalize:     d_betweenClasses[:,1:,cl,cl2] /= (mAbsDev_centroids[:,1:,cl]+mAbsDev_centroids[:,:-1,cl])/2
        else:
            d_betweenClasses[:,:,cl,cl2] = distance_riemann(run_centroids[:,:,cl], run_centroids[:,:,cl2])
            if doNormalize:     d_betweenClasses[:,:,cl,cl2] /= (mAbsDev_centroids[:,:,cl]+mAbsDev_centroids[:,:,cl2])/2

In [66]:
center_point = np.eye(run_centroids.shape[-1])

angles_to_zero = np.full((run_centroids.shape[0], run_centroids.shape[1], len(classes)), np.nan)
classes_angles = np.full((run_centroids.shape[0], run_centroids.shape[1]), np.nan)
for run_idx in np.ndindex(angles_to_zero.shape):
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = ref_ForAngle[run_idx[0]]
    vec = run_centroids[run_idx]
    angles_to_zero[run_idx[0], run_idx[1], run_idx[2]], _ = mrtrf.angle_between_matrices(base, vec, center_point)

# angles_to_zero = mrtrf.evaluate_negative_angles(angles_to_zero, run_centroids, center_point, positiveCen=run_centroids[0,0,0], positiveAngl=angles_to_zero[0,0,0])
angles_to_otherClass = np.full((run_centroids.shape[0], run_centroids.shape[1]), np.nan)
for run_idx in np.ndindex(angles_to_otherClass.shape[:2]):
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = run_centroids[run_idx[0], run_idx[1], 0]
    vec = run_centroids[run_idx[0], run_idx[1], 1]
    angles_to_otherClass[run_idx[0], run_idx[1]], _ = mrtrf.angle_between_matrices(base, vec, center_point)
from scipy.io import savemat
# savemat(f'{pathData}completeResults_20192020_t_betwCl_angles.mat', {'angles_to_otherClass': angles_to_otherClass})

angles_to_before = np.zeros((run_centroids.shape[0], run_centroids.shape[1], len(classes)))
for run_idx in np.ndindex(angles_to_before.shape):
    if run_idx[1] == 0:
        continue
    #base = run_centroids[tan_idx[0],0,tan_idx[2]]
    base = run_centroids[run_idx[0], run_idx[1]-1, run_idx[2]]
    vec = run_centroids[run_idx]
    angles_to_before[run_idx[0], run_idx[1], run_idx[2]], _ = mrtrf.angle_between_matrices(base, vec, center_point)
    
# from scipy.io import savemat
# savemat(f'{pathData}completeResults_20192020_t_before_angles.mat', {'angles_to_before': angles_to_before})

In [82]:
rows = []
for nrun, band, task in np.ndindex((d_ToEye.shape[1], d_ToEye.shape[0], d_ToEye.shape[2])):
    rows.append({
        # 'nRun': nrun,
        # 'band': band,
        # 'task': task,
        'runId' : validRuns[nrun],
        # 'nDay': dayVector[run_starts[nrun]].astype(int),
        # 'dayLabel': dates[np.where(x_dates<=nrun)[0][-1]],
        
        'distance' : d_ToEye[band, nrun, task],
        'angle' : angles_to_zero[band, nrun, task],

        # 'btw_distance': d_betweenClasses[band, nrun, 0, 1],
        'btw_angle': angles_to_otherClass[band, nrun],

        # 'wthin_distance': d_betweenClasses[band, nrun, task, task],
        'wth_angle': angles_to_before[band, nrun, task],
     
    })

winterResultsDf = pd.DataFrame(rows)

In [ ]:
copyResults = resultsDf.copy()
copyResults = copyResults[copyResults.runId<winterRun]

copyWinterRes = winterResultsDf.copy()
copyWinterRes = copyWinterRes[copyWinterRes.runId>=winterRun]

winter = pd.merge(copyResults, copyWinterRes, how='outer')
winter = winter[['distance', 'angle', 'btw_angle', 'wth_angle']]

for col in winter.columns:
    winter.rename(columns={col: 'blk_'+col}, inplace=True)

resultsDf = pd.concat([resultsDf, winter], axis=1)


In [ ]:
# saveName = 'results_user_2019.csv'
# resultsDf.to_csv(f"/home/palatella/workspace/cybathlon_results/{saveName}", index=False)

# PLOTS

In [10]:
stop_training = ['10/09/2020','03/10/2023','14/10/2024']  
stop_idx = np.array([next((x_dates[i] for i, s in enumerate(dates) if s == pattern), -1) for pattern in stop_training])
recalibrations = days_modelCreation[1:]
rec_idx = np.array([next((x_dates[i] for i, s in enumerate(dates) if s == pattern), -1) for pattern in recalibrations])

In [27]:
stop_training = ['10/09/2020','03/10/2023','14/10/2024']  
stop_idx = np.array([next((x_dates[i] for i, s in enumerate(dates) if s == pattern), -1) for pattern in stop_training])
recalibrations = days_modelCreation[1:]
rec_idx = np.array([next((x_dates[i] for i, s in enumerate(dates) if s == pattern), -1) for pattern in recalibrations])

idx_stop_plot = d_ToEye.shape[1]
idx_start_plot = 0

t_stop_idx = stop_idx[(stop_idx >= idx_start_plot) & (stop_idx < idx_stop_plot)] - idx_start_plot
t_rec_idx = rec_idx[(rec_idx >= idx_start_plot) & (rec_idx<idx_stop_plot)] - idx_start_plot
t_x_dates = x_dates[(x_dates >= idx_start_plot) & (x_dates<idx_stop_plot)] - idx_start_plot
t_x_accuracy = x_accuracy[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)] - idx_start_plot
                  
idx_stopRec = stop_idx[np.where((stop_idx >= idx_start_plot) & (stop_idx < idx_stop_plot))]-idx_start_plot
idx_recal = rec_idx[np.where((rec_idx >= idx_start_plot) & (rec_idx<idx_stop_plot))]-idx_start_plot

# std = std_centroids[:,idx_start_plot:idx_stop_plot]*5
# point_size=std,

saveFigures = False

In [ ]:
plt.close('all')

distance = d_ToEye[:,idx_start_plot:idx_stop_plot]
angl = angles_to_zero[:,idx_start_plot:idx_stop_plot]
# clss_dist = np.diagonal(d_betweenClasses[:,idx_start_plot:idx_stop_plot], axis1=-2, axis2=-1)
clss_dist = d_betweenClasses[:,idx_start_plot:idx_stop_plot, 0, 1]
withinClass_dist = np.diagonal(d_betweenClasses[:,idx_start_plot:idx_stop_plot], axis1=-2, axis2=-1)
# # # savemat(f'{pathData}completeResults_20192020_btwWith_clss_distance.mat', {'btwClss': clss_dist, 'withClss': withinClass_dist})


# max_value = np.max(distance)
max_distance = 1.2
max_angle = 2
max_dist = 1
min_angle = 0

fig = rplt.plot_centroids_movement(distance, classes, dates=dates[(x_dates >= idx_start_plot) & (x_dates < idx_stop_plot)], x_dates=t_x_dates, 
                                   accuracy=run_accuracy[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)], x_accuracy=t_x_accuracy, rejection=run_rejection[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)],
                                   max_value=max_distance, title='Distance to center (eye matrix) over days', stop_idx=t_stop_idx, rec_idx=t_rec_idx, bandranges=bandranges)

filename = 'eyeDistance_20192020.svg' if not doLogBandPower else 'eyeDistance_logBandPower_20192020.svg'
if doNormalize: filename = filename.replace('.svg', '_normalized.svg')
if not applyLaplacian: filename = filename.replace('.svg', '_noLap.svg')
if saveFigures:
    print('Saving figure to ' + filename)
    fig.savefig(f'{pathImages}{filename}', bbox_inches='tight', dpi=1000, format='svg')


fig = rplt.plot_centroids_movement(angl, classes, dates=dates[(x_dates >= idx_start_plot) & (x_dates<idx_stop_plot)], x_dates=t_x_dates, 
                                   accuracy=run_accuracy[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)], x_accuracy=t_x_accuracy, rejection=run_rejection[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)],
                                   max_value=max_angle, min_value=min_angle,   title='Centroids angles over days', stop_idx=t_stop_idx, rec_idx=t_rec_idx, bandranges=bandranges)

filename = 'angleVariation_20192020.svg' if not doLogBandPower else 'angleVariation_logBandPower_20192020.svg'
if doNormalize: filename = filename.replace('.svg', '_normalized.svg')
if not applyLaplacian: filename = filename.replace('.svg', '_noLap.svg')
if saveFigures:
    print('Saving figure to ' + filename)
    fig.savefig(f'{pathImages}{filename}', bbox_inches='tight', dpi=1000, format='svg')

t = np.cos(angl)
fig = rplt.plot_centroids_movement(t, classes, dates=dates[(x_dates >= idx_start_plot) & (x_dates<idx_stop_plot)], x_dates=t_x_dates, 
                                   accuracy=run_accuracy[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)], x_accuracy=t_x_accuracy, rejection=run_rejection[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)],
                                   max_value=1, min_value=-1,   title='Centroids cosine angles over days', stop_idx=t_stop_idx, rec_idx=t_rec_idx, bandranges=bandranges)

filename = 'cos_angleVariation_20192020.svg' if not doLogBandPower else 'cos_angleVariation_logBandPower_20192020.svg'
if doNormalize: filename = filename.replace('.svg', '_normalized.svg')
if not applyLaplacian: filename = filename.replace('.svg', '_noLap.svg')
if saveFigures:
    print('Saving figure to ' + filename)
    fig.savefig(f'{pathImages}{filename}', bbox_inches='tight', dpi=1000, format='svg')

t = np.sin(angl)
fig = rplt.plot_centroids_movement(t, classes, dates=dates[(x_dates >= idx_start_plot) & (x_dates<idx_stop_plot)], x_dates=t_x_dates, 
                                   accuracy=run_accuracy[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)], x_accuracy=t_x_accuracy, rejection=run_rejection[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)],
                                   max_value=max_angle, min_value=min_angle,   title='Centroids sine angles over days', stop_idx=t_stop_idx, rec_idx=t_rec_idx, bandranges=bandranges)

filename = 'sin_angleVariation_20192020.svg' if not doLogBandPower else 'sin_angleVariation_logBandPower_20192020.svg'
if doNormalize: filename = filename.replace('.svg', '_normalized.svg')
if not applyLaplacian: filename = filename.replace('.svg', '_noLap.svg')
if saveFigures:
    print('Saving figure to ' + filename)
    fig.savefig(f'{pathImages}{filename}', bbox_inches='tight', dpi=1000, format='svg')



fig = rplt.polarPlot_centroids(distance, angl, bandranges=bandranges, idx_stopRec=idx_stopRec, idx_recal=idx_recal, max_distance=max_distance)

filename = 'polarPlot_20192020.svg' if not doLogBandPower else 'polarPlot_logBandPower_20192020.svg'
if doNormalize: filename = filename.replace('.svg', '_normalized.svg')
if not applyLaplacian: filename = filename.replace('.svg', '_noLap.svg')
if saveFigures:
    print('Saving figure to ' + filename)
    fig.savefig(f'{pathImages}{filename}', bbox_inches='tight', dpi=1000, format='svg')

fig = rplt.plot_centroids_movement(clss_dist, classes, dates=dates[(x_dates >= idx_start_plot) & (x_dates < idx_stop_plot)], x_dates=t_x_dates, 
                                   accuracy=run_accuracy[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)], x_accuracy=t_x_accuracy, rejection=run_rejection[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)],
                                   max_value=max_dist, title='Class Distinctiveness', stop_idx=t_stop_idx, rec_idx=t_rec_idx, bandranges=bandranges)


filename = 'class_distinctiveness_20192020.svg' if not doLogBandPower else 'class_distinctiveness_logBandPower_20192020.svg'
if doNormalize: filename = filename.replace('.svg', '_normalized.svg')
if not applyLaplacian: filename = filename.replace('.svg', '_noLap.svg')
if saveFigures:
    print('Saving figure to ' + filename)
    fig.savefig(f'{pathImages}{filename}', bbox_inches='tight', dpi=1000, format='svg')

fig = rplt.plot_centroids_movement(withinClass_dist, classes, dates=dates[(x_dates >= idx_start_plot) & (x_dates < idx_stop_plot)], x_dates=t_x_dates, 
                                   accuracy=run_accuracy[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)], x_accuracy=t_x_accuracy, rejection=run_rejection[(x_accuracy >= idx_start_plot) & (x_accuracy<idx_stop_plot)],
                                   max_value=max_dist, title='Distance from previous centroid', stop_idx=t_stop_idx, rec_idx=t_rec_idx, bandranges=bandranges)


filename = 'previousCentroid_distance_20192020.svg' if not doLogBandPower else 'previousCentroid_distance_logBandPower_20192020.svg'
if doNormalize: filename = filename.replace('.svg', '_normalized.svg')
if not applyLaplacian: filename = filename.replace('.svg', '_noLap.svg')
if saveFigures:
    print('Saving figure to ' + filename)
    fig.savefig(f'{pathImages}{filename}', bbox_inches='tight', dpi=1000, format='svg')
